# Task 4: Time Series Analysis (Fedez vs. Fabri Fibra)

## 1. Setup and Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import warnings
from tslearn.clustering import TimeSeriesKMeans
from tslearn.preprocessing import TimeSeriesScalerMeanVariance
from tslearn.shapelets import LearningShapelets
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from collections import Counter

warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('seaborn-v0_8-muted')

## 2. Data Loading & Helper Functions
We need to identify which files belong to Fedez and which to Fabri Fibra.

In [ ]:
MP3_FOLDER = "../../Downloads/fedez_fibra"
DATASETS_FOLDER = "../datasets"

# Artist Mapping (Verified)
ARTIST_MAPPING = {
    "07024718": "Fedez",
    "25707984": "Fabri Fibra"
}

def get_artist_from_filename(filename):
    # Format: ART<id> - TR<id>.mp3
    try:
        artist_id = filename.split(' - ')[0].replace('ART', '')
        return ARTIST_MAPPING.get(artist_id, "Unknown")
    except:
        return "Unknown"

files = [f for f in os.listdir(MP3_FOLDER) if f.endswith('.mp3')]
print(f"Found {len(files)} MP3 files.")

# Filter only Fedez and Fabri Fibra (in case there are others)
target_files = [f for f in files if get_artist_from_filename(f) in ["Fedez", "Fabri Fibra"]]
print(f"Processing {len(target_files)} relevant files.")

## 3. Visualize Audio Features (Prototype)
Before processing everything, let's visualize the waveform and spectrogram of one song from each artist.

In [ ]:
def visualize_audio(file_path, title="Audio Visualization"):
    # Load audio (using a lower sample rate for speed if needed, e.g., sr=22050)
    # Loading first 30 seconds for visualization
    y, sr = librosa.load(file_path, sr=22050, duration=30)
    
    plt.figure(figsize=(14, 8))
    
    # Waveform
    plt.subplot(2, 1, 1)
    librosa.display.waveshow(y, sr=sr)
    plt.title(f"{title} - Waveform")
    
    # Mel Spectrogram
    plt.subplot(2, 1, 2)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    S_dB = librosa.power_to_db(S, ref=np.max)
    librosa.display.specshow(S_dB, sr=sr, x_axis='time', y_axis='mel')
    plt.colorbar(format='%+2.0f dB')
    plt.title(f"{title} - Mel Spectrogram")
    
    plt.tight_layout()
    plt.show()

# Visualize one random file from Fedez and one from Fabri Fibra
fedez_sample = next((f for f in target_files if get_artist_from_filename(f) == "Fedez"), None)
fibra_sample = next((f for f in target_files if get_artist_from_filename(f) == "Fabri Fibra"), None)

if fedez_sample:
    print("Visualizing Fedez sample:", fedez_sample)
    visualize_audio(os.path.join(MP3_FOLDER, fedez_sample), title=f"Fedez - {fedez_sample}")

if fibra_sample:
    print("Visualizing Fabri Fibra sample:", fibra_sample)
    visualize_audio(os.path.join(MP3_FOLDER, fibra_sample), title=f"Fabri Fibra - {fibra_sample}")

## 4. Feature Extraction
We will extract **MFCCs (Mel-frequency cepstral coefficients)** to represent each song as a time series.
Parameters:
- `sr=22050`: Standard sample rate.
- `n_mfcc=13`: Number of coefficients (standard for speech/music).
- `duration=30`: We'll take a 30-second slice from the middle of the song to standardize length and reduce processing time. If the song is shorter, we'll pad it.

In [ ]:
DURATION = 30 # seconds
SAMPLE_RATE = 22050
N_MFCC = 13

def extract_features(file_path):
    try:
        # Load audio
        # Start loading from 30s mark to avoid intros, or just load and slice
        # We load full file to find middle, or just load first X mins.
        # Let's try loading with offset to get the "meat" of the song.
        # But first let's just load it. `librosa.load` is fast enough for 30s.
        
        # Get duration first (fast)
        duration_total = librosa.get_duration(path=file_path)
        offset = max(0, (duration_total - DURATION) / 2)
        
        y, sr = librosa.load(file_path, sr=SAMPLE_RATE, offset=offset, duration=DURATION)
        
        # Pad if shorter than DURATION
        target_len = int(DURATION * SAMPLE_RATE)
        if len(y) < target_len:
            y = np.pad(y, (0, target_len - len(y)))
        elif len(y) > target_len:
            y = y[:target_len]
            
        # Extract MFCCs
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
        
        # Result shape: (n_mfcc, time_steps)
        # Transpose to (time_steps, n_mfcc) for tslearn if needed, or keep as is.
        # tslearn usually expects (n_samples, sz, d)
        return mfcc.T
        
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# Lists to store data
X_list = []
y_labels = []
filenames_processed = []

print("Starting feature extraction...")
# We will process a subset if too many, e.g. first 50 of each for speed during development
# But let's try to do all if possible.
for f in tqdm(target_files):
    path = os.path.join(MP3_FOLDER, f)
    features = extract_features(path)
    
    if features is not None:
        X_list.append(features)
        y_labels.append(get_artist_from_filename(f))
        filenames_processed.append(f)

# Convert to numpy arrays
X = np.array(X_list)
y = np.array(y_labels)

print(f"Feature extraction complete.")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

## 5. Clustering (Task 4.1)
We use **K-Means** with **DTW (Dynamic Time Warping)** or Euclidean distance to group songs.
Since standard DTW is slow, we might use `softdtw` or keep Euclidean if the alignment is already good (since we took middle slice).

In [ ]:
# Preprocess data: Scale time series to have mean 0 and variance 1
scaler = TimeSeriesScalerMeanVariance()
X_scaled = scaler.fit_transform(X)

# Train TimeSeriesKMeans
# k=2 (Fedez vs Fabri Fibra, ideally)
print("Training KMeans...")
km = TimeSeriesKMeans(n_clusters=2, metric="euclidean", max_iter=10, random_state=42)
labels = km.fit_predict(X_scaled)

# Analyze Results
print("Cluster counts:", Counter(labels))

# Check correlation with Artist
df_res = pd.DataFrame({"Filename": filenames_processed, "Artist": y, "Cluster": labels})
print("\nCluster distribution by Artist:")
print(pd.crosstab(df_res['Artist'], df_res['Cluster']))

### Visualize Centroids (Motifs)
The cluster centroids represent the "average" song in that cluster, effectively capturing the **motif** or common pattern of that group.

In [ ]:
plt.figure(figsize=(12, 6))
for i, centroid in enumerate(km.cluster_centers_):
    plt.subplot(1, 2, i + 1)
    # Visualize first MFCC coefficient (Energy/Loudness approx) and maybe another
    plt.plot(centroid[:, 0], label='MFCC 0 (Loudness)')
    plt.plot(centroid[:, 1], label='MFCC 1 (Timbre)', alpha=0.7)
    plt.title(f"Cluster {i} Centroid")
    plt.legend()
plt.tight_layout()
plt.show()

### Anomaly Detection
We identify anomalies as the songs that are furthest from their cluster centroid.

In [ ]:
# Calculate distances to assigned centroid
distances = []
for i in range(len(X_scaled)):
    cluster_idx = labels[i]
    centroid = km.cluster_centers_[cluster_idx]
    # Euclidean distance
    dist = np.linalg.norm(X_scaled[i] - centroid)
    distances.append(dist)

df_res['Distance'] = distances

# Top 5 Anomalies
anomalies = df_res.sort_values('Distance', ascending=False).head(5)
print("Top 5 Anomalies (Furthest from centroid):")
print(anomalies)

## 6. Shapelet Extraction (Task 4.2)
We use **LearningShapelets** to find shapelets (discriminative subsequences) that best separate Fedez from Fabri Fibra.

In [ ]:
# Prepare labels for classification
le = LabelEncoder()
y_enc = le.fit_transform(y)

# Define Shapelet Model
print("Training Shapelet Model (this may take a while)...")
# Only using a few shapelets for speed demonstration
shp_clf = LearningShapelets(
    n_shapelets_per_size={50: 3}, 
    max_iter=50,
    verbose=1,
    optimizer="adam",
    scale=False,
    random_state=42
)

shp_clf.fit(X_scaled, y_enc)
print("Shapelet training complete.")

# Predictions
y_pred = shp_clf.predict(X_scaled)
print(f"Accuracy: {accuracy_score(y_enc, y_pred):.2f}")

### Visualize Learning Shapelets
We visualize the learned shapelets and where they match in the time series.

In [ ]:
predicted_locations = shp_clf.transform(X_scaled)
shapelet_vis_count = min(3, len(shp_clf.shapelets_as_time_series_))

plt.figure(figsize=(15, 5 * shapelet_vis_count))
for i in range(shapelet_vis_count):
    s = shp_clf.shapelets_as_time_series_[i]
    # Find the best match across all samples
    # transform returns distances. We want to find sample with min distance to this shapelet.
    # Actually transform returns min distances.
    
    # Let's just plot the shapelet itself first (first dimension)
    plt.subplot(shapelet_vis_count, 1, i + 1)
    plt.plot(s[:, 0], label='Shapelet Dim 0 (Loudness)')
    if s.shape[1] > 1:
        plt.plot(s[:, 1], label='Shapelet Dim 1 (Timbre)', alpha=0.6)
    plt.title(f"Shapelet {i}")
    plt.legend()

plt.tight_layout()
plt.show()